# Notebook 01: OpenSky Flight Departures

## Note
I built this local notebook as OpenSky do not allow IP addresses associated with Google Cloud Platform (GCP), or any other major cloud service provider, to query their endpoints.  

The work performed here is additive to what I'm doing on Google Colab, and much of the code here originates from there. Google Colab remains my first choice for notebook-based research. 
Google Colab Notebook link: https://colab.research.google.com/drive/1fzTssli75DMSBWKRzdaBCISDKLF8VVJz?usp=sharing  

The output directory (`output/01-notebook`) has been added to .gitignore.   
   
To run this notebook, you will first need to sign up and receive API credentials from OpenSky (opensky-network.org). Then you must create an .env with your credentials assigned to two variables, `OPENSKY_API_CLIENT_ID` and `OPENSKY_API_KEY`.   

In [7]:
import json
import os
import urllib.error
import urllib.parse
import urllib.request
from datetime import datetime, timedelta, timezone
from pathlib import Path

from dotenv import load_dotenv

In [8]:
# OpenSky now only allow access to their API with the OAuth2 client credentials flow.

OPENSKY_API_CLIENT_ID = os.environ["OPENSKY_API_CLIENT_ID"]
OPENSKY_API_KEY = os.environ["OPENSKY_API_KEY"]
TOKEN_URL = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"

In [9]:
# Airport ICAO ID constants
DUBLIN_AIRPORT_ICAO_CODE = "EIDW"
CORK_AIRPORT_ICAO_CODE = "EICK"
SHANNON_AIRPORT_ICAO_CODE = "EINN"

IRELAND_AIRPORTS_ICAO_CODES = [DUBLIN_AIRPORT_ICAO_CODE, CORK_AIRPORT_ICAO_CODE, SHANNON_AIRPORT_ICAO_CODE]

In [10]:
# SMALL ADAPTATION FROM SOURCE: https://openskynetwork.github.io/opensky-api/rest.html

def get_access_token():
    data = urllib.parse.urlencode({
        "grant_type": "client_credentials",
        "client_id": OPENSKY_API_CLIENT_ID,
        "client_secret": OPENSKY_API_KEY,
    }).encode()

    request = urllib.request.Request(TOKEN_URL, data=data, method="POST")
    request.add_header("Content-Type", "application/x-www-form-urlencoded")

    with urllib.request.urlopen(request, timeout=10) as resp:
        access_token = json.load(resp)["access_token"]
        
        return access_token

def test_access_fn():
    token = get_access_token()
    if token:
        print("Token:" + token[:4])
    else:
        print("No token!")

test_access_fn()

Token:eyJh


In [11]:
# ADAPTED FROM SOURCE: https://openskynetwork.github.io/opensky-api/rest.html
# TOKEN COST: 30
def fetch_24h_departures(airport_icao_code):
  current_time = datetime.now(timezone.utc)
  time_delta_time = current_time - timedelta(days=1)

  begin_timestamp= int(time_delta_time.timestamp())
  end_timestamp = int(current_time.timestamp())
  
  params = urllib.parse.urlencode({
      'airport': airport_icao_code,
      'begin': begin_timestamp,
      'end': end_timestamp
  })

  url = f"https://opensky-network.org/api/flights/departure?{params}"
  request = urllib.request.Request(url)
  request.add_header("Authorization", f"Bearer {get_access_token()}")
  
  try:
    with urllib.request.urlopen(request, timeout=10) as response:
      departures = json.load(response)
      print("Credits remaining:", response.headers.get("X-Rate-Limit-Remaining"))

      output_time = datetime.now().strftime("%Y-%m-%d-%H%M%S")

      output_dir = Path("output/01-notebook")
      output_dir.mkdir(parents=True, exist_ok=True)
    
      output_file = f"{airport_icao_code}-24h-dep-{output_time}.json"
      output_path = output_dir/output_file

      with open(output_path, "w") as f:
        json.dump(departures, f, indent=2)
  except urllib.error.HTTPError as http_e:
    print(f"HTTP Error: {http_e.code}")
  except urllib.error.URLError as url_e:
    print(f"URL Error: {url_e.reason}")
  except Exception as err:
    print(f"An error occurred: {err}")

fetch_24h_departures(DUBLIN_AIRPORT_ICAO_CODE);

Credits remaining: 3910


In [12]:
# ADAPTED FROM SOURCE: https://openskynetwork.github.io/opensky-api/rest.html
# TOKEN COST: 30
def fetch_6h_departures(airport_icao_code):
  current_time = datetime.now(timezone.utc)
  time_delta_time = current_time - timedelta(hours=6)

  begin_timestamp= int(time_delta_time.timestamp())
  end_timestamp = int(current_time.timestamp())
  
  params = urllib.parse.urlencode({
      'airport': airport_icao_code,
      'begin': begin_timestamp,
      'end': end_timestamp
  })
  url = f"https://opensky-network.org/api/flights/departure?{params}"
  
  request = urllib.request.Request(url)
  request.add_header("Authorization", f"Bearer {get_access_token()}")
  
  try:
    with urllib.request.urlopen(request, timeout=10) as response:
      departures = json.load(response)
      print("Credits remaining:", response.headers.get("X-Rate-Limit-Remaining"))

      output_time = datetime.now().strftime("%Y-%m-%d-%H%M%S")

      output_dir = Path("output/01-notebook")
      output_dir.mkdir(parents=True, exist_ok=True)
    
      output_file = f"{airport_icao_code}-6h-dep-{output_time}.json"
      output_path = output_dir/output_file

      with open(output_path, "w") as f:
        json.dump(departures, f, indent=2)
  except urllib.error.HTTPError as http_e:
    print(f"HTTP Error: {http_e.code}")
  except urllib.error.URLError as url_e:
    print(f"URL Error: {url_e.reason}")
  except Exception as err:
    print(f"An error occurred: {err}")

fetch_6h_departures(DUBLIN_AIRPORT_ICAO_CODE);

Credits remaining: 3880
